<a href="https://colab.research.google.com/github/mrdbourke/pytorch-deep-learning/blob/main/extras/exercises/05_pytorch_going_modular_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. PyTorch Going Modular Exercises

Welcome to the 05. PyTorch Going Modular exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.

> **Note:** There may be more than one solution to each of the exercises, don't worry too much about the *exact* right answer. Try to write some code that works first and then improve it if you can.

## Resources and solutions

* These exercises/solutions are based on [section 05. PyTorch Going Modular](https://www.learnpytorch.io/05_pytorch_going_modular/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.

**Solutions:**

Try to complete the code below *before* looking at these.

* See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/ijgFhMK3pp4).
* See an example [solutions notebook for these exercises on GitHub](https://github.com/mrdbourke/pytorch-deep-learning/blob/main/extras/solutions/05_pytorch_going_modular_exercise_solutions.ipynb).

## 1. Turn the code to get the data (from section 1. Get Data) into a Python script, such as `get_data.py`.

* When you run the script using `python get_data.py` it should check if the data already exists and skip downloading if it does.
* If the data download is successful, you should be able to access the `pizza_steak_sushi` images from the `data` directory.

In [1]:
%mkdir going_modular

In [2]:
# YOUR CODE HERE
%%writefile going_modular/get_data.py
"""
Contains functionality for getting data.
"""
import requests
import zipfile

from pathlib import Path

data_path = Path("data")
image_path = data_path / "pizza_steak_sushi"

if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print("Downloading pizza, steak, sushi data...")
    f.write(request.content)

with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping pizza, steak, sushi data...")
    zip_ref.extractall(image_path)

Writing going_modular/get_data.py


In [3]:
# Example running of get_data.py
!python going_modular/get_data.py

Did not find data/pizza_steak_sushi directory, creating one...
Unzipping pizza, steak, sushi data...


## 2. Use [Python's `argparse` module](https://docs.python.org/3/library/argparse.html) to be able to send the `train.py` custom hyperparameter values for training procedures.
* Add an argument flag for using a different:
  * Training/testing directory
  * Learning rate
  * Batch size
  * Number of epochs to train for
  * Number of hidden units in the TinyVGG model
    * Keep the default values for each of the above arguments as what they already are (as in notebook 05).
* For example, you should be able to run something similar to the following line to train a TinyVGG model with a learning rate of 0.003 and a batch size of 64 for 20 epochs: `python train.py --learning_rate 0.003 batch_size 64 num_epochs 20`.
* **Note:** Since `train.py` leverages the other scripts we created in section 05, such as, `model_builder.py`, `utils.py` and `engine.py`, you'll have to make sure they're available to use too. You can find these in the [`going_modular` folder on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/going_modular/going_modular).

In [4]:
import requests
from pathlib import Path

module_path = Path("going_modular")
module_path.mkdir(parents=True, exist_ok=True)

print(f"Ensuring {module_path} directory exists.")

base_url = "https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/going_modular/going_modular/"

# List of module files to download
module_files = ["data_setup.py", "engine.py", "model_builder.py", "utils.py"]

print("Downloading going_modular modules...")
for file_name in module_files:
    file_url = base_url + file_name
    target_file_path = module_path / file_name

    # Check if file already exists to avoid re-downloading
    if target_file_path.is_file():
        print(f"{file_name} already exists, skipping download.")
        continue

    print(f"Downloading {file_name}...")
    try:
        request = requests.get(file_url)
        request.raise_for_status()

        with open(target_file_path, "wb") as f:
            f.write(request.content)
        print(f"Downloaded {file_name} to {target_file_path}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {file_name}: {e}")
        print(f"Could not download {file_name} from {file_url}")

print("Finished downloading going_modular modules.")

Ensuring going_modular directory exists.
Downloaded data_setup.py to going_modular/data_setup.py
Downloaded engine.py to going_modular/engine.py
Downloaded model_builder.py to going_modular/model_builder.py
Downloaded utils.py to going_modular/utils.py
Finished downloading going_modular modules.


In [5]:
# YOUR CODE HERE
%%writefile going_modular/train.py
"""
Trains a PyTorch image classification model using device-agnostic code with ability to define hyperparameter values.
"""
import argparse
import torch

from torchvision import transforms
from pathlib import Path

from data_setup import create_dataloaders
from model_builder import TinyVGG
from engine import train
from utils import save_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

parser = argparse.ArgumentParser(description="Define hyperparameters for training loop.")

parser.add_argument("--num_epochs", default=10, type=int, help="the number of epochs to train for")
parser.add_argument("--batch_size", default=32, type=int, help="number of data samples per batch")
parser.add_argument("--hidden_units", default=10, type=int, help="number of hidden units in hidden layers")
parser.add_argument("--learning_rate", default=0.001, type=float, help="learning rate to use for model")

args = parser.parse_args()

NUM_EPOCHS = args.num_epochs
BATCH_SIZE = args.batch_size
HIDDEN_UNITS = args.hidden_units
LEARNING_RATE = args.learning_rate

train_path = Path("data/pizza_steak_sushi/train")
test_path = Path("data/pizza_steak_sushi/test")

data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

train_dataloader, test_dataloader, class_names = create_dataloaders(
    train_dir=train_path,
    test_dir=test_path,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

model = TinyVGG(
    input_shape=3,
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)
).to(device)

results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=torch.nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=LEARNING_RATE),
    epochs=NUM_EPOCHS,
    device=device
)

save_model(
    model=model,
    target_dir="models",
    model_name="05_going_modular_script_mode_tinyvgg_model.pth"
)


Writing going_modular/train.py


In [6]:
# Example running of train.py
!python going_modular/train.py --num_epochs 5 --batch_size 128 --hidden_units 128 --learning_rate 0.0003

  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | train_loss: 1.0963 | train_acc: 0.3859 | test_loss: 1.0902 | test_acc: 0.3333
 20% 1/5 [00:02<00:09,  2.27s/it]Epoch: 2 | train_loss: 1.0759 | train_acc: 0.3715 | test_loss: 1.0763 | test_acc: 0.3733
 40% 2/5 [00:03<00:05,  1.85s/it]Epoch: 3 | train_loss: 1.0526 | train_acc: 0.4760 | test_loss: 1.0319 | test_acc: 0.4533
 60% 3/5 [00:05<00:03,  1.72s/it]Epoch: 4 | train_loss: 0.9949 | train_acc: 0.5971 | test_loss: 1.0226 | test_acc: 0.4133
 80% 4/5 [00:06<00:01,  1.48s/it]Epoch: 5 | train_loss: 0.9256 | train_acc: 0.5619 | test_loss: 1.0050 | test_acc: 0.4667
100% 5/5 [00:07<00:00,  1.51s/it]
[INFO] Saving model to: models/05_going_modular_script_mode_tinyvgg_model.pth


## 3. Create a Python script to predict (such as `predict.py`) on a target image given a file path with a saved model.

* For example, you should be able to run the command `python predict.py some_image.jpeg` and have a trained PyTorch model predict on the image and return its prediction.
* To see example prediction code, check out the [predicting on a custom image section in notebook 04](https://www.learnpytorch.io/04_pytorch_custom_datasets/#113-putting-custom-image-prediction-together-building-a-function).
* You may also have to write code to load in a trained model.

In [7]:
# YOUR CODE HERE
%%writefile going_modular/predict.py
import torch
import argparse

from PIL import Image
from torchvision import transforms
from pathlib import Path

from model_builder import TinyVGG

parser = argparse.ArgumentParser(description="Predict on a target image with a trained model.")

parser.add_argument("--image", help="target image filepath to predict on")
parser.add_argument("--model_path", default="models/05_going_modular_script_mode_tinyvgg_model.pth", type=str, help="target model to use for prediction filepath")

args = parser.parse_args()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load image
image = Image.open(args.image)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])
image = transform(image).unsqueeze(0).to(device)

# Load model
model = TinyVGG(input_shape=3, hidden_units=128, output_shape=3).to(device)
model.load_state_dict(torch.load(args.model_path))

# Predict
model.eval()
with torch.inference_mode():
    pred_logits = model(image)
    pred_probs = torch.softmax(pred_logits, dim=1)
    pred_label = torch.argmax(pred_probs, dim=1)

print(f"Predicted Probs: {pred_probs} | Predicted Class: {pred_label}")

Writing going_modular/predict.py


In [8]:
# Example running of predict.py
!python going_modular/predict.py --image data/pizza_steak_sushi/test/sushi/175783.jpg

Predicted Probs: tensor([[0.4527, 0.1566, 0.3907]], device='cuda:0') | Predicted Class: tensor([0], device='cuda:0')
